In [1]:
# %%
import os
import numpy as np
import h5py

rng = np.random.default_rng(42)

file = 'CSTR_InputVectors.h5'   # output file

N = 100_000                     # samples per series (t + 9 fault variables -> ~1,000,000 combined)
t = np.arange(N, dtype=float)   # timesteps [s]

# --- Fault definitions: ramp up -> hold -> ramp down; count=0 disables a fault ---
# deltas below are a stress-test pass (~2-3x prior magnitudes), scaled per-channel rather than
# uniformly: E_R only ever slows the reaction so it was pushed hardest (3x) with no stability
# cost; U_Ac/T_CF directly erode cooling-jacket heat transfer and were kept closer to 2-2.5x
# since they already push the coolant valve (Qc, bounded by Cv2*sqrt(dPc)) into saturation for
# a large fraction of each hold - verified by isolated step-hold tests to settle, not run away.
FAULTS = {
    1: dict(name='E_R',    baseline=8750.0,  delta=270.0,      variance=0.1, sign=0, count=4),
    2: dict(name='U_Ac',   baseline=5e4/60,  delta=-225.0,     variance=0.1, sign=0, count=3),
    3: dict(name='T_Bias', baseline=0.0,     delta=5.0,        variance=0.5, sign=1, count=0),  # disabled - set count>0 to re-enable
    4: dict(name='T_F',    baseline=320.0,   delta=0.35*90*2.5, variance=0.2, sign=1, count=4),
    5: dict(name='C_F',    baseline=1.0,     delta=0.0006*90*2.5, variance=0.3, sign=1, count=3),
    6: dict(name='T_CF',   baseline=300.0,   delta=0.1*150*2,  variance=0.2, sign=1, count=4),
    7: dict(name='Q_F',    baseline=100/60,  delta=10/60*3,    variance=0.1, sign=1, count=4),
    8: dict(name='dP',     baseline=50.0,    delta=5.0,        variance=0.4, sign=1, count=0),  # disabled - set count>0 to re-enable
    9: dict(name='dPc',    baseline=25.0,    delta=2.5,        variance=0.4, sign=1, count=0),  # disabled - set count>0 to re-enable
}

DURATION_RANGE = (2000, 4000)   # total samples per fault occurrence (~2-4% of N)
RAMP_FRACTION = 0.15            # fraction of an event's duration spent ramping up, and again ramping down
MIN_GAP = 300                    # minimum baseline samples between events

# --- Schedule non-overlapping fault events (strict: only one active at a time) ---
events = []
for fid, cfg in FAULTS.items():
    events += [fid] * cfg['count']
rng.shuffle(events)

durations = rng.integers(DURATION_RANGE[0], DURATION_RANGE[1] + 1, size=len(events))
total_fault_time = int(durations.sum())
total_steady_time = N - total_fault_time
num_gaps = len(events) + 1

if total_steady_time < MIN_GAP * num_gaps:
    raise ValueError("Not enough room for the requested faults with MIN_GAP steady time between them - reduce counts/durations or increase N.")

gap_shares = rng.dirichlet(np.ones(num_gaps))
gaps = np.floor(gap_shares * (total_steady_time - MIN_GAP * num_gaps)).astype(int) + MIN_GAP
gaps[-1] += total_steady_time - gaps.sum()   # fix rounding remainder

schedule = []   # (fault_id, start, end, duration)
pos = gaps[0]
for i, fid in enumerate(events):
    dur = int(durations[i])
    schedule.append((fid, pos, pos + dur, dur))
    pos += dur + gaps[i + 1]

assert pos == N

# --- Build the output vectors: ramp up / hold / ramp down --------------------
vectors = {fid: cfg['baseline'] * np.ones(N) for fid, cfg in FAULTS.items()}
flags = {fid: np.zeros(N) for fid in FAULTS}     # 1 for the whole event (ramps + hold)
holds = {fid: np.zeros(N) for fid in FAULTS}     # 1 only during the held steady-state portion

event_info = []   # (fid, up_dur, hold_dur, down_dur, magnitude)
for fid, start, end, dur in schedule:
    cfg = FAULTS[fid]
    magnitude = cfg['delta'] * (1 + rng.uniform(0, cfg['variance']))
    if cfg['sign'] == 1:
        magnitude *= rng.choice([1, -1])

    up_dur = max(1, round(dur * RAMP_FRACTION))
    down_dur = max(1, round(dur * RAMP_FRACTION))
    hold_dur = dur - up_dur - down_dur

    up = np.linspace(0, magnitude, up_dur)
    hold = np.full(hold_dur, magnitude)
    down = np.linspace(magnitude, 0, down_dur)
    perturbation = np.concatenate([up, hold, down])

    vectors[fid][start:end] = cfg['baseline'] + perturbation
    flags[fid][start:end] = 1
    holds[fid][start + up_dur:start + up_dur + hold_dur] = 1
    event_info.append((fid, up_dur, hold_dur, down_dur, magnitude))

# AR1 measurement noise, applied to F3-F9
def AR1(n, p, s, rng):
    ar = np.zeros(n)
    for i in range(1, n):
        ar[i] = ar[i - 1] * p + rng.normal(0, s)
    return ar / 100 + 1

p, s = 0.9999, 0.005
for fid in (3, 4, 5, 6, 7, 8, 9):
    vectors[fid] = vectors[fid] * AR1(N, p, s, rng)

# --- Overlap check (should be zero) ------------------------------------------
overlap_count = np.vstack(list(flags.values())).sum(axis=0)
assert overlap_count.max() <= 1, "Faults overlap - scheduling bug"
print(f"Max simultaneous faults: {int(overlap_count.max())} (should be 1)")

nominal_time = N - overlap_count.sum()
hold_time = np.vstack(list(holds.values())).sum()
transient_time = overlap_count.sum() - hold_time

print(f"Distinct steady states: {1 + len(events)} (1 nominal + {len(events)} fault holds, each a different magnitude)")
print(f"Time at nominal baseline:      {nominal_time / N:.1%}")
print(f"Time held at another steady state: {hold_time / N:.1%}")
print(f"Time in transition (ramping):  {transient_time / N:.1%}")
print(f"Total fault events: {len(events)}")
for fid, cfg in FAULTS.items():
    if cfg['count'] > 0:
        info = [e for e in event_info if e[0] == fid]
        print(f"  F{fid} ({cfg['name']}): {cfg['count']} events, (ramp_up, hold, ramp_down) = {[(u, h, d) for _, u, h, d, _ in info]}")

# --- Save --------------------------------------------------------------------
# creates dataset if missing, overwrites if present
def save_vector(filename, name, data):
    with h5py.File(filename, 'a') as f:
        if name in f:
            del f[name]
        f.create_dataset(name, data=data, chunks=True, compression='gzip')
        print(f"Saved dataset '{name}' with shape {data.shape}")

if os.path.exists(file):
    os.remove(file)

save_vector(file, 't', t)
for fid in FAULTS:
    save_vector(file, f'F{fid}', vectors[fid].reshape(-1, 1))
    save_vector(file, f'F{fid}.plt', flags[fid])


Max simultaneous faults: 1 (should be 1)
Distinct steady states: 23 (1 nominal + 22 fault holds, each a different magnitude)
Time at nominal baseline:      33.6%
Time held at another steady state: 46.5%
Time in transition (ramping):  19.9%
Total fault events: 22
  F1 (E_R): 4 events, (ramp_up, hold, ramp_down) = [(566, 2644, 566), (319, 1489, 319), (527, 2462, 527), (320, 1495, 320)]
  F2 (U_Ac): 3 events, (ramp_up, hold, ramp_down) = [(328, 1528, 328), (548, 2560, 548), (510, 2381, 510)]
  F4 (T_F): 4 events, (ramp_up, hold, ramp_down) = [(421, 1963, 421), (368, 1718, 368), (466, 2177, 466), (434, 2023, 434)]
  F5 (C_F): 3 events, (ramp_up, hold, ramp_down) = [(433, 2021, 433), (383, 1787, 383), (591, 2760, 591)]
  F6 (T_CF): 4 events, (ramp_up, hold, ramp_down) = [(493, 2302, 493), (547, 2552, 547), (435, 2031, 435), (489, 2285, 489)]
  F7 (Q_F): 4 events, (ramp_up, hold, ramp_down) = [(464, 2163, 464), (558, 2601, 558), (350, 1630, 350), (406, 1897, 406)]
Saved dataset 't' with shap